# Introduction

This notebook aims to understand and visualize real-time GTFS (RT) data reported from the TTC.
Specifically we would like to visualize Route 29, a bus route that is known to have significantly high trip times.

We would like to compare the state of the transit network (with respect to Route 29) to the scheduled state (through GTFS static data provided
by the TTC)

You can find the exact data source used in this project below: 
GTFS Static data: https://open.toronto.ca/dataset/merged-gtfs-ttc-routes-and-schedules/
GTFS RT data: https://gtfsrt.ttc.ca/

# Data Modeling

Before we dig into the data itself, we will start off by understanding the inherent structure of the data (the data model). The GTFS  specifciation is a global standard used to structure transit data reported by transit agencies. Specifically the GTFS specifies an array of CSV text files each representing different types of transit information (i.e. route/trip information, schedule information, geo-shape information etc.). Each CSV text file relate to each other through a relational schema.

A relational UML schema representing the GTFS RT data reported by the TTC can be found below:

![GTFS_RealTime_Relational_Diagram](GTFS_RT_Relational_Diagram.jpeg)


It is likely that GTFS static data is needed to provide complementary information to the GTFS RT data, so the relational schema for the GTFS static data is also provided below:

![GTFS_STATIC_Relational_Model](GTFS_STATIC_Relational_Model.jpeg)

# Data Collection and Preprocessing

GTFS-RT data provides insight into the instantaneous state of the transit system. 

For example, trip_update provides stoptime event data from the current vehicle position/stop onwards and veichle_position measures the instantaneous speed, location and position of a veichle going through a trip/route.

Therefore in order to have sufficient data for analysis, we will need to query TTC's RT GTFS api multiple times to get historical data; and then calculate some running average to evalaute the state of the transit system.

To implement this, we will need to break the problem down into 2 steps:

a. create a data collection and preprocessing script that continously collects, formats and processes queried data and saves it into a unified database 

b. create a data analysis script that analyzes the recorded data in the database

## Data Collection

We will start off by performing a. 

Real-time data was collected using GET requests to the TTC GTFS-RT api

In [1]:
import requests
import subprocess
import json
import pandas as pd
import re
from datetime import datetime

alerts_url = "https://gtfsrt.ttc.ca/alerts/all?format=text"
trip_updates_url = "https://gtfsrt.ttc.ca/trips/update?format=text"
veichle_positions_url = "https://gtfsrt.ttc.ca/vehicles/position?format=text"

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36"
}

/Users/devrajsolanki/Documents/TAL/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
def collect_data(trip_updates_url, veichle_positions_url):
    # reading trip updates data
    trip_updates_response = requests.get(trip_updates_url, headers=headers)
    trip_updates_text = trip_updates_response.text

    # reading vehicle position data
    veichle_positions_response = requests.get(veichle_positions_url, headers=headers)
    veichle_positions_text = veichle_positions_response.text
    return trip_updates_text, veichle_positions_text


In [3]:
# Additonally static data will be loaded 

import pandas as pd

stops = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/stops.txt")
routes = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/routes.txt")
trips = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/trips.txt")
stop_times = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/stop_times.txt")
shapes = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/shapes.txt")
calendar = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/calendar.txt")
agency = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/agency.txt")
calendar_dates = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/calendar_dates.txt")
route_types = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/route_types.txt")
feed_info = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/feed_info.txt")

/var/folders/b2/y50nhjkn7554xcj0cffkg3m80000gn/T/ipykernel_18716/3023753728.py:7: DtypeWarning: Columns (4,7) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/trips.txt")
/var/folders/b2/y50nhjkn7554xcj0cffkg3m80000gn/T/ipykernel_18716/3023753728.py:8: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/stop_times.txt")


## Data Formatting

We will format the queried data according to a modified schema based on the GTFS RT schema diagram

![Modified_GTFS_RT_Relational_Diagram.jpeg](Modified_GTFS_RT_Relational_Diagram.jpeg)

In [4]:
# made with claude code

# takes in raw trip_update text data and formats it into a dataframe according to the modified GTFS RT schema above
def modify_trip_updates(response_text: str) -> list[dict]:
    text = response_text

    # gets all information in each "block" of information w/r to each entity
    def blocks(src, key):
        for m in re.finditer(rf'\b{key}\s*\{{', src):
            depth, i = 0, m.end() - 1
            while i < len(src):
                depth += (src[i] == '{') - (src[i] == '}')
                if depth == 0: yield src[m.end():i]; break
                i += 1

    # some blocks dont repeat i.e. veichle id or trip information w/r to each entity, so we just get the first match or return none
    def first(src, key):
        return next(blocks(src, key), None)

    # given a block of text, we return the value in the key:value pairs stored within the feed data (i.e. stop_sequence: val )
    def val(src, key):
        m = re.search(rf'\b{key}\s*:\s*"?([^"\n]+?)"?\s*$', src or '', re.M)
        return m.group(1) if m else None

    # get timestamp of feed
    feed_timestamp = val(first(text, 'header'), 'timestamp')

    # record each stop update within a trip update as one dicionary or record in the TripUpdate dataframe
    records = []
    for entity in blocks(text, 'entity'):
        tu = first(entity, 'trip_update')
        if not tu: continue
        trip = first(tu, 'trip')
        veh  = first(tu, 'vehicle')
        base = {
            'id':                         val(entity, 'id'),
            'trip_id':                    val(trip,   'trip_id'),
            'trip_schedule_relationship': val(trip,   'schedule_relationship'),
            'route_id':                   val(trip,   'route_id'),
            'vehicle_id':                 val(veh,    'id'),
            'timestamp':                  val(tu,     'timestamp'),
        }
        for stu in blocks(tu, 'stop_time_update'):
            records.append({**base,
                'stop_sequence':              val(stu, 'stop_sequence'),
                'stop_id':                    val(stu, 'stop_id'),
                'stop_schedule_relationship': val(stu, 'schedule_relationship'),
                'arrival_time':               val(first(stu, 'arrival'),   'time'),
                'departure_time':             val(first(stu, 'departure'), 'time'),
            })
    return pd.DataFrame(records), feed_timestamp



In [5]:
# made with claude code

# takes in raw vehicle_positions text data and formats it into a dataframe according to the modified GTFS RT schema above

def modify_vehicle_positions(response_text: str) -> list[dict]:
    text = response_text
 
    def blocks(src, key):
        for m in re.finditer(rf'\b{key}\s*\{{', src):
            depth, i = 0, m.end() - 1
            while i < len(src):
                depth += (src[i] == '{') - (src[i] == '}')
                if depth == 0: yield src[m.end():i]; break
                i += 1
 
    def first(src, key):
        return next(blocks(src, key), None)
 
    def val(src, key):
        m = re.search(rf'\b{key}\s*:\s*"?([^"\n]+?)"?\s*$', src or '', re.M)
        return m.group(1) if m else None
 
    feed_timestamp = val(first(text, 'header'), 'timestamp')

    records = []
    for entity in blocks(text, 'entity'):
        vp = first(entity, 'vehicle')
        if not vp: continue
        trip = first(vp, 'trip')
        pos  = first(vp, 'position')
        veh  = first(vp, 'vehicle')      # nested vehicle { id: ... }
        records.append({
            'id':                    val(entity, 'id'),
            'latitude':              val(pos,    'latitude'),
            'longitude':             val(pos,    'longitude'),
            'bearing':               val(pos,    'bearing'),
            'speed':                 val(pos,    'speed'),
            'timestamp':             val(vp,     'timestamp'),
            'vehicle_id':            val(veh,    'id'),
            'occupancy_status':      val(vp,     'occupancy_status'),
            'current_stop_sequence': val(vp,     'current_stop_sequence'),
            'current_status':        val(vp,     'current_status'),
            'stop_id':               val(vp,     'stop_id'),
            'trip_id':               val(trip,   'trip_id'),
            'schedule_relationship': val(trip,   'schedule_relationship'),
            'route_id':              val(trip,   'route_id'),
        })
    return pd.DataFrame(records), feed_timestamp


## Data Processing

In [6]:
# processing for trip_updates

def process_trip_updates(df):
    df = df.copy()
    
    # replace No_Data schedule relationships with NaN for arrival/departure times
    df.loc[df['stop_schedule_relationship'] == 'NO_DATA', ['arrival_time', 'departure_time']] = "no data provided"

    # combine arrival_time and departure_time into a single column
    df['arrival/departure_time'] = df['arrival_time'].combine_first(df['departure_time'])

    # drop arrival and departure time columns
    df = df.drop(columns=['arrival_time', 'departure_time'])
    
    # replace missing vehicle_id with NaN
    df['vehicle_id'] = df['vehicle_id'].replace('', pd.NA)

    # convert timestamp from POSIX to datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'].astype(int), unit='s')

    return df.reset_index(drop=True)


In [7]:
# preprocessing for vehicle_positions

def process_vehicle_positions(df):
    df = df.copy()
    
    # impute missing occupancy_status with most common value
    df['occupancy_status'] = df['occupancy_status'].fillna(df['occupancy_status'].mode()[0])

    # drop rows where route_id is missing (when route_id is missing, so is trip_id, trip_schedule_relationship etc. [trip_id, stop_id, status])
    df = df.dropna(subset=['route_id'])

    # convert speed from m/s to km/hr
    df['speed_km/hr'] = df['speed'].astype(float) * 3.6

    # change timestamp from POSIX to current time
    df['timestamp'] = pd.to_datetime(df['timestamp'].astype(int), unit='s')

    return df.reset_index(drop=True)



In [8]:
# query data, format it and save the timestamped data for safekeeping
trip_updates_txt, veichle_positions_txt = collect_data(trip_updates_url, veichle_positions_url)

trip_updates, tu_feed_time = modify_trip_updates(trip_updates_txt)
vehicle_positions, vp_feed_time  = modify_vehicle_positions(veichle_positions_txt)

trip_updates = process_trip_updates(trip_updates)
vehicle_positions = process_vehicle_positions(vehicle_positions)


trip_updates_dict = {datetime.fromtimestamp(int(tu_feed_time)): trip_updates}
vehicle_positions_dict = {datetime.fromtimestamp(int(vp_feed_time)): vehicle_positions}

# Data Analysis